In [ ]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
!pip install transformers datasets tensorflow pdfplumber --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 137.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 142.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 134.3 MB/s eta 0:00:00


In [ ]:
!pip install keras-hub

In [11]:
from datasets import load_dataset

In [12]:
dataset=load_dataset("abisee/cnn_dailymail","3.0.0")

README.md:   0%|          | 0.00/15.6k [00:00<?, ?B/s]

3.0.0/train-00000-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  257MB            

3.0.0/train-00000-of-00003.parquet: downloading bytes:           |  0.00B            

3.0.0/train-00001-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  257MB            

3.0.0/train-00001-of-00003.parquet: downloading bytes:           |  0.00B            

...

3.0.0/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 30.0MB            

3.0.0/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

In [13]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 287113
    })
    validation: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 13368
    })
    test: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 11490
    })
})


In [14]:
print(dataset["train"][0]["article"][:500])
print("\n SUMMARY")
print(dataset["train"][0]["highlights"])

LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won't cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his cash away on fast cars, drink and celebrity parties. "I don't plan to be one of those people who, as s

 SUMMARY
Harry Potter star Daniel Radcliffe gets £20M fortune as he turns 18 Monday .
Young actor says he has no plans to fritter his cash away .
Radcliffe's earnings from first five Potter films have been held in trust fund .


In [15]:
small_train=dataset["train"].select(range(500))
small_val=dataset["validation"].select(range(100))
print(small_train)
print(small_val)

Dataset({
    features: ['article', 'highlights', 'id'],
    num_rows: 500
})
Dataset({
    features: ['article', 'highlights', 'id'],
    num_rows: 100
})


In [ ]:
import keras_hub
preprocessor=keras_hub.models.BartSeq2SeqLMPreprocessor.from_preset(
    "bart_base_en",
    encoder_sequence_length=1024,
    decoder_sequence_length=128
)

In [ ]:
model=keras_hub.models.BartSeq2SeqLM.from_preset(
    "bart_base_en",
    preprocessor=preprocessor
)

In [ ]:
def to_dictionary_format(hf_dataset):
  return {
      "encoder_text" : list(hf_dataset["article"])
      ,"decoder_text" : list(hf_dataset["highlights"])
  }

In [ ]:
train_dict=to_dictionary_format(small_train)
val_dict=to_dictionary_format(small_val)

In [ ]:
train_ds = tf.data.Dataset.from_tensor_slices(train_dict).batch(4)
val_ds = tf.data.Dataset.from_tensor_slices(val_dict).batch(4)

print(train_ds)
print(val_ds)

<_BatchDataset element_spec={'encoder_text': TensorSpec(shape=(None,), dtype=tf.string, name=None), 'decoder_text': TensorSpec(shape=(None,), dtype=tf.string, name=None)}>
<_BatchDataset element_spec={'encoder_text': TensorSpec(shape=(None,), dtype=tf.string, name=None), 'decoder_text': TensorSpec(shape=(None,), dtype=tf.string, name=None)}>


In [ ]:
model.compile(
    weighted_metrics=["accuracy"],
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-5)
)

In [ ]:
model.fit(
    train_ds,
    epochs=1,
    validation_data=val_ds
)

125/125 ━━━━━━━━━━━━━━━━━━━━ 205s 869ms/step - accuracy: 0.5358 - loss: 1.0828 - val_accuracy: 0.5580 - val_loss: 0.7633


In [ ]:
sample_article=dataset["test"][0]["article"]
output=model.generate(
    {"encoder_text" : sample_article , "decoder_text" : ""},
    max_length=128
)
print("SUmmary \n")
print(output)

SUmmary 

NEW: New designation makes Palestinians the 123rd member of the International Criminal Court .
The decision marks a move toward greater justice, human rights and peace .
The new designation means Palestinians can join the court for the first time .
The move is the first in the ICC's 123rd year .


In [ ]:
print(dataset["test"][0]["highlights"])

Membership gives the ICC jurisdiction over alleged crimes committed in Palestinian territories since last June .
Israel and the United States opposed the move, which could open the door to war crimes investigations against Israelis .


In [ ]:
train_subset=dataset["train"].select(range(10000))
validation_subset=dataset["validation"].select(range(1000))
train_dict={
    "encoder_text" : list(train_subset["article"]),
    "decoder_text": list(train_subset["highlights"])
}
val_dic={
    "encoder_text": list(validation_subset["article"]),
    "decoder_text" : list(validation_subset["highlights"])
}
train_ds=tf.data.Dataset.from_tensor_slices(train_dict).batch(4)
val_ds=tf.data.Dataset.from_tensor_slices(val_dict).batch(4)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
early_stop=EarlyStopping(
  monitor="val_loss",
  patience=1,
  restore_best_weights=True
)

In [ ]:
model.fit(
    train_ds,
    epochs=2,
    validation_data=val_ds,
    callbacks=[early_stop]
)

Epoch 1/2
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 1874s 748ms/step - accuracy: 0.5904 - loss: 0.9603 - val_accuracy: 0.5722 - val_loss: 0.7226
Epoch 2/2
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 1876s 749ms/step - accuracy: 0.6592 - loss: 0.7331 - val_accuracy: 0.5661 - val_loss: 0.7428


In [ ]:
model.save_to_preset("docecho_bart_model")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

model.save_to_preset("/content/drive/MyDrive/docecho_bart_model")

Mounted at /content/drive


In [ ]:
!pip install rouge_score evaluate

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.9 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24986 sha256=ba57daeb6ca491d02f7fc7d673e6d4f34c5b7ef0b5a27c21cd552eb440252532
  Stored in directory: /root/.cache/pip/wheels/44/af/da/5ffc433e2786f0b1a9c6f458d5fb8f611d8eb332387f18698f
Successfully built rouge_score


In [16]:
test_subset=dataset["test"].select(range(50))

In [17]:
article=list(test_subset["article"])

In [18]:
refrence_highlights=list(test_subset["highlights"])
generated_summaries=[]
batch_size=5
for i in range(0,len(article),batch_size):
  batch=article[i:i+batch_size]
  outputs=model.generate(
      {"encoder_text" : batch , "decoder_text": [""]*batch_size},
      max_length=128,

  )
  generated_summaries.extend(outputs)

NameError: name 'model' is not defined

In [ ]:
print(len(generated_summaries))
print(generated_summaries[0])

50
Palestinians accepted their membership of the International Criminal Court .
They are not a member of the ICC, which gives them jurisdiction over alleged crimes in Palestinians' territories .
Palestinian Foreign Minister Riad al-Malki says it is a move toward greater justice .
Palestinian Authority says it is a step closer to ending a long era of impunity, injustice .


In [ ]:
import json

with open("/content/drive/MyDrive/docecho_eval_outputs.json", "w") as f:
    json.dump({"generated": list(generated_summaries), "reference": reference_summaries}, f)

NameError: name 'reference_summaries' is not defined

In [19]:


from google.colab import drive
drive.mount('/content/drive')

import json
with open("/content/drive/MyDrive/docecho_eval_outputs.json") as f:
    data = json.load(f)

generated_summaries = data["generated"]
reference_summaries = data["reference"]

print(len(generated_summaries), len(reference_summaries))
print(generated_summaries[0])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
50 50
Palestinians accepted their membership of the International Criminal Court .
They are not a member of the ICC, which gives them jurisdiction over alleged crimes in Palestinians' territories .
Palestinian Foreign Minister Riad al-Malki says it is a move toward greater justice .
Palestinian Authority says it is a step closer to ending a long era of impunity, injustice .


In [1]:
!pip install -q --upgrade evaluate rouge_score

In [4]:
import evaluate
rouge=evaluate.load("rouge")
results=rouge.compute(
    predictions=generated_summaries,
    references=reference_summaries
)

In [6]:
for key,value in results.items():
  print (f"{key} : {value}")

rouge1 : 0.2961462427303685
rouge2 : 0.09088898671037839
rougeL : 0.2108008113071807
rougeLsum : 0.2735443058894391


In [20]:
import keras_hub
base_preprocessor = keras_hub.models.BartSeq2SeqLMPreprocessor.from_preset(
    "bart_base_en",
    encoder_sequence_length=1024,
    decoder_sequence_length=128,
)

base_model = keras_hub.models.BartSeq2SeqLM.from_preset(
    "bart_base_en",
    preprocessor=base_preprocessor,
)

base_generated_summaries = []
batch_size = 5

for i in range(0, len(article), batch_size):
    batch = article[i:i+batch_size]
    outputs = base_model.generate(
        {"encoder_text": batch, "decoder_text": [""] * len(batch)},
        max_length=128
    )
    base_generated_summaries.extend(outputs)

print(len(base_generated_summaries))

50


In [21]:
import evaluate
rouge=evaluate.load("rouge")
results=rouge.compute(
    predictions=base_generated_summaries,
    references=reference_summaries
)

In [22]:
for key,items in results.items():
  print(f"{key} : {value}")

rouge1 : 0.2735443058894391
rouge2 : 0.2735443058894391
rougeL : 0.2735443058894391
rougeLsum : 0.2735443058894391
